In [25]:
import os
import requests
import sys
sys.path.append('../..')
from bs4 import BeautifulSoup
from scraper import fetch_website_contents , fetch_website_links
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import json 

In [5]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [6]:
try:
    r = requests.get("http://localhost:11434")
    print("Ollama is running:", r.content.decode())
except Exception:
    print("Ollama is NOT running. Open a terminal and run: ollama serve")

Ollama is running: Ollama is running


In [7]:
links=fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

In [8]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [10]:
print(get_links_user_prompt("https://edwarddonner.com"))



Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [11]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model = "llama3.2" , 
        messages = [{"role": "system" , "content": link_system_prompt },{"role": "user" , "content" : get_links_user_prompt(url)}],
        response_format ={"type":"json_object"}
        )

      
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [12]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'company page', 'url': 'https://huggingface.co/'},
  {'type': 'blog page', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'changelog page',
   'url': 'https://docs.huggingface.co/vision/latest/changelog.html'},
  {'type': 'organization page',
   'url': 'https://www.zhihu.com/org/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand/'},
  {'type': 'press contact link', 'url': 'mailto:press@huggingface.co'}]}

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))


## Landing Page:

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 600,000 enrolled across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m su

In [28]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short ,humorous,entertaining,witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [29]:
def get_brochure_user_prompt(company_name , url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_brochure_user_prompt("EdDonner" , "https://edwarddonner.com")

'\nYou are looking at a company called: EdDonner\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula.io\n. I was previously founder and CEO of AI startup untapt,\nacquired in 2021\n, and a Managing Director at JPMorgan.\nI will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, an

In [ ]:
def create_brochure(company_name , url):
    response = ollama.chat.completions.create(
        model = "llama3.2" ,
        messages = [
            {"role":"system" , "content":brochure_system_prompt} ,
            {"role":"user" , "content" : get_brochure_user_prompt(company_name , url)} 
            
        ] 
        
    )

    result=response.choices[0].message.content
    display(Markdown(result))


In [ ]:
create_brochure("EdDonner" , "https://edwarddonner.com")


# EdDonner: Elevating Human Prosperity through AI

At EdDonner, our mission is to help people discover their potential and pursue their reason for being. We believe that by unlocking human prosperity, we can create a better world for all.

## Our Story

We were founded by Ed, a renowned AI expert with a passion for finding real-world solutions to complex problems. With his expertise in Generative AI and machine learning, Ed Co-Founded Nebula.io, an innovative startup dedicated to sourcing, understanding, engaging, and managing talent using cutting-edge technologies.

## patented Matching Technology

Our proprietary model matches people with roles that align with their strengths, interests, and values, providing a more accurate and efficient process than traditional keywords-based matching. This results in better job satisfaction, increased productivity, and improved overall well-being for our clients.

## Ikigai: Finding Your Reason

EdDonner's long-term goal is to help individuals discover their Ikigai – that sweet spot where passion meets purpose. By providing personalized career guidance and support, we empower people to find fulfilling roles and contribute to a more prosperous society.

## Impact

At EdDonner, we take pride in making a positive impact:

* **77% of people don't consider themselves inspired or engaged at work** – we're committed to changing this narrative.
* **Nebula.io's patented model matches people with roles 3x faster than traditional methods**.
* **Ed's Udemy courses have reached 400,000 students worldwide**, providing accessible knowledge and skills training.

## Join Our Community

If you're interested in shaping the future of human prosperity through AI, join us!

* Follow Ed on LinkedIn | Twitter | Facebook to stay updated on our latest insights and breakthroughs.
* Explore our comprehensive curriculum and courses at [www.edwarddonner.com](http://www.edwarddonner.com).

Let's work together to unlock humanity's full potential!

In [24]:
create_brochure("EdDonner" , "https://edwarddonner.com")


**EdDonner: Revolutionizing Talent Acquisition and AI Innovation**

At EdDonner, we're passionate about harnessing the power of Generative AI to transform the way we find, understand, engage, and manage talent. Our mission is to empower individuals to discover their potential and pursue their reason for being, driving a higher level of human prosperity.

**Our Story**

Founded by Ed Donner, our co-founder and CTO, Nebula.io has been at the forefront of AI innovation since 2013. Previously, Ed founded untapt, an AI startup acquired in 2021, and worked as a Managing Director at JPMorgan. With a background in lecturing and educating others on LLMs and Agents, he created best-selling Udemy courses with 600,000 enrolled users worldwide.

**What We Do**

Our patented model uses Generative AI to match people with roles, offering greater accuracy and speed than traditional methods. No keywords required! Our goal is to help individuals find their dream jobs, leading to increased motivation and success.

**Our Values**

• **Ikigai**: We believe that people have a reason for being, and we aim to help them find it.
• **Continuous Learning**: We're committed to staying at the forefront of AI innovation, with a focus on practical applications.
• **Community Building**: Our events, courses, and resources foster a supportive community where individuals can grow and learn together.

**Join Our Community**

Stay up-to-date with our latest news, research, and resources. Join us for live events, webinars, or interact with our digital avatar to dive deeper into AI and talent acquisition topics.

**Get in Touch**

Contact us at [ed@edwarddonner.com](mailto:ed@edwarddonner.com) or visit our website to explore our patented model and discover how EdDonner can help you achieve your goals.

In [26]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [30]:
stream_brochure("EdDonner" ,"https://edwarddonner.com")

**Welcome to EdDonner**

We're a team of passionate individuals on a mission to make a difference in the world of Artificial Intelligence and Talent Management. Our founders' vision is inspired by the concept of Ikigai – finding happiness and fulfillment through meaningful work.

**Our Journey So Far**
Co-founded by renowned AI expert, Ed, who has a passion for teaching and exploring the exciting world of LLMs and Agents. We've spent years developing innovative solutions to real-world problems, including talent management.

**What Sets Us Apart**
At EdDonner, we believe in making AI accessible to everyone. Our patented model helps recruiters source, understand, engage, and manage talent with unprecedented accuracy. No more relying on keywords alone!

**Our Values**

* **Empowering Talent**: We want people to discover their potential and pursue their reason for being.
* **Community First**: We're dedicated to building a supportive community of like-minded individuals who share our vision.
* **Lifelong Learning**: Continuously learning and improving is essential in the AI space. Stay ahead with our cutting-edge courses and resources!

**Get Involved**

Ready to join our mission? Explore our curriculum, Outsmart arena, or become part of our Proficient AI Engineer Directory. The future of work starts here.

**Stay Connected**

Follow us on social media or visit [www.edwarddonner.com](http://www.edwarddonner.com) for more updates and insights into the world of AI talent management.

**We Can't Wait to Help You Discover Your Dream Job!